# Content-Based Movie Recommendations

This notebook explores recommendations based on movie genres.

The collaborative-filtering model compared movies using patterns in
users' ratings. This model will compare movies using their attributes.

We will use the same training, validation, and test split and the same
candidate movies as the earlier experiments. This makes their
validation results comparable.

The test set will remain reserved for the final evaluation.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(data_dir / "movies.csv")
ratings = pd.read_csv(data_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

dates = ratings["rated_at"]

train_cutoff = dates.quantile(0.60)
validation_cutoff = dates.quantile(0.80)

train_ratings = ratings.loc[
    dates <= train_cutoff
].copy()

validation_ratings = ratings.loc[
    (dates > train_cutoff) & (dates <= validation_cutoff)
].copy()

test_ratings = ratings.loc[
    dates > validation_cutoff
].copy()

print("Training rows:", len(train_ratings))
print("Validation rows:", len(validation_ratings))
print("Test rows:", len(test_ratings))

Training rows: 60502
Validation rows: 20167
Test rows: 20167


Select a data set of movies with training ratings that includes the genre

In [2]:
content_movies = movies.loc[
    movies["movieId"].isin(train_ratings["movieId"]),
    ["movieId", "title", "genres"]
].copy()

content_movies = content_movies.sort_values(
    "movieId"
).reset_index(drop=True)

print("Candidate movies:", len(content_movies))

content_movies.head()

Candidate movies: 6233


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Create a multi-hot encoding data frame where genres are columns and movies are rows. A 1 means the movie belongs to that genre and a zero means it dfoesn't

In [3]:
def split_genres(genre_text):
    """Convert a movie's genre string into a list of genre labels."""
    if genre_text == "(no genres listed)":
        return []

    return genre_text.split("|")


content_movies["genre_list"] = content_movies["genres"].apply(
    split_genres
)

content_movies[["movieId", "title", "genre_list"]].head()

,movieId,title,genre_list
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


Scikit-learn’s MultiLabelBinarizer converts these lists into a binary matrix.

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

genre_encoder = MultiLabelBinarizer()

genre_values = genre_encoder.fit_transform(
    content_movies["genre_list"]
)

genre_matrix = pd.DataFrame(
    genre_values,
    index=content_movies["movieId"],
    columns=genre_encoder.classes_
)

print("Genre matrix shape:", genre_matrix.shape)

genre_matrix.iloc[:5, :8]

Genre matrix shape: (6233, 19)


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama
movieId,,,,,,,,
1,0,1,1,1,1,0,0,0
2,0,1,0,1,0,0,0,0
3,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,1
5,0,0,0,0,1,0,0,0


calculate genre similarity

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

toy_story_genres = genre_matrix.loc[[1]]
jumanji_genres = genre_matrix.loc[[2]]

genre_similarity = cosine_similarity(
    toy_story_genres,
    jumanji_genres
)[0, 0]

print("Toy Story genre count:", toy_story_genres.sum(axis=1).iloc[0])
print("Jumanji genre count:", jumanji_genres.sum(axis=1).iloc[0])
print("Genre cosine similarity:", genre_similarity)

Toy Story genre count: 5
Jumanji genre count: 3
Genre cosine similarity: 0.7745966692414834


Compare Toy Story with every candidate movie

In [6]:
seed_movie_id = 1

genre_similarity_scores = cosine_similarity(
    genre_matrix.loc[[seed_movie_id]],
    genre_matrix
)[0]

similar_genre_movies = pd.DataFrame({
    "movieId": genre_matrix.index,
    "similarity": genre_similarity_scores
}).merge(
    content_movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
    validate="one_to_one"
)

similar_genre_movies = similar_genre_movies.loc[
    similar_genre_movies["movieId"] != seed_movie_id
].sort_values(
    ["similarity", "movieId"],
    ascending=[False, True]
).reset_index(drop=True)

similar_genre_movies.head(10)

,movieId,similarity,title,genres
0,2294,1.000000,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
1,3114,1.000000,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2,3754,1.000000,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3,4016,1.000000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
4,4886,1.000000,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
5,45074,1.000000,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6,53121,1.000000,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
7,673,0.912871,Space Jam (1996),Adventure|Animation|Children|Comedy|Fantasy|Sc...
8,4306,0.912871,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...
9,36397,0.912871,Valiant (2005),Adventure|Animation|Children|Comedy|Fantasy|War


Use genre and user training likes to produce reccomendations

In [7]:
example_user_id = 1

user_history = train_ratings.loc[
    train_ratings["userId"] == example_user_id
]

liked_movie_ids = user_history.loc[
    user_history["rating"] >= 4.0,
    "movieId"
]

genre_similarities = cosine_similarity(
    genre_matrix.loc[liked_movie_ids],
    genre_matrix
)

user_genre_scores = pd.Series(
    genre_similarities.mean(axis=0),
    index=genre_matrix.index,
    name="score"
)

print("Movies rated in training:", len(user_history))
print("Movies liked in training:", len(liked_movie_ids))
print("Similarity array shape:", genre_similarities.shape)

Movies rated in training: 232
Movies liked in training: 200
Similarity array shape: (200, 6233)


exclude previously rated movies and rank the candidates. Select the recommendations

We exclude all movies the user already rated, then sort by:
1. Highest average similarity score.
2. Highest training rating count when scores tie.
3. Lowest movie ID when both tie.
Popularity does not contribute to the numerical score; it breaks ties. The score is a measure of genre similarity, rather than a predicted star rating or probability that the user will like the movie.

In [8]:
genre_recommendations = content_movies.loc[
    ~content_movies["movieId"].isin(user_history["movieId"]),
    ["movieId", "title", "genres"]
].copy()

genre_recommendations["score"] = (
    genre_recommendations["movieId"].map(user_genre_scores)
)

training_rating_counts = train_ratings["movieId"].value_counts()

genre_recommendations["training_rating_count"] = (
    genre_recommendations["movieId"].map(training_rating_counts)
)

genre_recommendations = genre_recommendations.sort_values(
    ["score", "training_rating_count", "movieId"],
    ascending=[False, False, True]
).head(10).reset_index(drop=True)

genre_recommendations

,movieId,title,genres,score,training_rating_count
0,55116,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,0.426350,1
1,5657,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,0.421825,1
2,6990,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,0.421825,1
3,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,0.417705,2
4,4818,Extreme Days (2001),Action|Adventure|Comedy|Drama,0.416454,1
5,42015,Casanova (2005),Action|Adventure|Comedy|Drama|Romance,0.403711,2
6,2890,Three Kings (1999),Action|Adventure|Comedy|Drama|War,0.399067,39
7,6503,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller,0.389882,15
8,8968,After the Sunset (2004),Action|Adventure|Comedy|Crime|Thriller,0.389882,4
9,60074,Hancock (2008),Action|Adventure|Comedy|Crime|Fantasy,0.385263,5


we’ll turn this calculation into a reusable function so we can evaluate it for every validation user.



In [9]:
movie_popularity = (
    train_ratings.groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(
        movies[["movieId", "title"]],
        on="movieId",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        ["training_rating_count", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

movie_popularity.head(10)

,movieId,training_rating_count,title
0,296,201,Pulp Fiction (1994)
1,356,193,Forrest Gump (1994)
2,480,185,Jurassic Park (1993)
3,318,177,"Shawshank Redemption, The (1994)"
4,593,177,"Silence of the Lambs, The (1991)"
5,110,174,Braveheart (1995)
6,589,166,Terminator 2: Judgment Day (1991)
7,457,160,"Fugitive, The (1993)"
8,592,160,Batman (1989)
9,780,157,Independence Day (a.k.a. ID4) (1996)


The function:
1. Gets the user’s training history and identifies ratings of at least 4.
2. Excludes every movie the user already rated.
3. Calculates each candidate’s average similarity to the liked movies.
4. Sorts the candidates and returns the top n.

In [12]:
def recommend_genre_based(
    user_id,
    training_data,
    genre_matrix,
    popularity_ranking,
    n=10
):
    """Recommend by mean genre similarity, with popularity as a fallback."""

    user_history = training_data.loc[
        training_data["userId"] == user_id
    ]

    liked_movie_ids = user_history.loc[
        user_history["rating"] >= 4.0,
        "movieId"
    ]

    recommendations = popularity_ranking.loc[
        ~popularity_ranking["movieId"].isin(user_history["movieId"])
    ].copy()

    recommendations["score"] = 0.0

    if not liked_movie_ids.empty:
        similarities = cosine_similarity(
            genre_matrix.loc[liked_movie_ids],
            genre_matrix
        )

        scores = pd.Series(
            similarities.mean(axis=0),
            index=genre_matrix.index
        )

        recommendations["score"] = (
            recommendations["movieId"].map(scores).fillna(0.0)
        )

    return recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    ).head(n).reset_index(drop=True)

In [13]:
for example_user_id in [1, 2]:
    print("Recommendations for user:", example_user_id)

    display(
        recommend_genre_based(
            user_id=example_user_id,
            training_data=train_ratings,
            genre_matrix=genre_matrix,
            popularity_ranking=movie_popularity,
            n=10
        )
    )

Recommendations for user: 1


,movieId,training_rating_count,title,score
0,55116,1,"Hunting Party, The (2007)",0.426350
1,5657,1,Flashback (1990),0.421825
2,6990,1,The Great Train Robbery (1978),0.421825
3,4956,2,"Stunt Man, The (1980)",0.417705
4,4818,1,Extreme Days (2001),0.416454
5,42015,2,Casanova (2005),0.403711
6,2890,39,Three Kings (1999),0.399067
7,6503,15,Charlie's Angels: Full Throttle (2003),0.389882
8,8968,4,After the Sunset (2004),0.389882
9,60074,5,Hancock (2008),0.385263


Recommendations for user: 2


,movieId,training_rating_count,title,score
0,296,201,Pulp Fiction (1994),0.0
1,356,193,Forrest Gump (1994),0.0
2,480,185,Jurassic Park (1993),0.0
3,318,177,"Shawshank Redemption, The (1994)",0.0
4,593,177,"Silence of the Lambs, The (1991)",0.0
5,110,174,Braveheart (1995),0.0
6,589,166,Terminator 2: Judgment Day (1991),0.0
7,457,160,"Fugitive, The (1993)",0.0
8,592,160,Batman (1989),0.0
9,780,157,Independence Day (a.k.a. ID4) (1996),0.0


Now let’s measure how well this method retrieves liked validation movies.

In [14]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= 4.0
].copy()

evaluation_user_ids = validation_positives["userId"].unique()

genre_based_rows = []

for user_id in evaluation_user_ids:
    user_positive_movie_ids = validation_positives.loc[
        validation_positives["userId"] == user_id,
        "movieId"
    ]

    recommendations = recommend_genre_based(
        user_id=user_id,
        training_data=train_ratings,
        genre_matrix=genre_matrix,
        popularity_ranking=movie_popularity,
        n=10
    )

    hits = recommendations["movieId"].isin(
        user_positive_movie_ids
    ).sum()

    genre_based_rows.append({
        "userId": user_id,
        "positive_movie_count": len(user_positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(user_positive_movie_ids)
    })

genre_based_results = pd.DataFrame(genre_based_rows)

print("Users evaluated:", len(genre_based_results))
print(
    "Mean Recall@10:",
    genre_based_results["recall_at_10"].mean()
)

Users evaluated: 165
Mean Recall@10: 0.03580190341510486


In [15]:
genre_based_results["has_training_history"] = (
    genre_based_results["userId"].isin(train_ratings["userId"])
)

genre_group_summary = (
    genre_based_results.groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

genre_group_summary

,users,mean_recall_at_10
has_training_history,,
False,140,0.041847
True,25,0.001950


## Genre-based recommender: validation findings

The genre-based recommender represents each candidate movie using
19 binary genre features. Candidates are scored by their average
cosine similarity to movies the user rated at least 4.0 during training.

Previously rated movies are excluded. Popularity breaks tied scores
and provides a fallback when positive training history is unavailable.

Mean validation Recall@10 was:

- Overall: 3.580% across 165 users.
- Without training history: 4.185% across 140 users.
- With training history: 0.195% across 25 users.

The genre-based model underperformed both popularity and item-based
collaborative filtering. Collaborative filtering remains the strongest
method on validation so far.

Genre labels cannot distinguish movies with identical genre vectors.
The effects of averaging across varied preferences and favoring rare
genre combinations are possible explanations requiring further study.

This experiment provides a genre-only benchmark for testing richer
content features, including plot text. The test set remains unevaluated.